# SDNC — Predictive Coding Training on Colab

**Architecture:** Sparse Dynamic Neural Circuits with Predictive Coding

- Circuits predict next input → prediction error drives learning
- Global state persists as contextual prior
- Temporal stream — no isolated image processing

**Saves to Google Drive** — resume anytime.

In [ ]:
# === Cellule 1 — Setup ===
from google.colab import drive
drive.mount('/content/drive')

# Clone or update repo
import os
REPO_DIR = '/content/sdnc'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    # IMPORTANT: remplacer par ton URL de repo
    !git clone https://github.com/YOUR_USER/sdnc.git {REPO_DIR}

!pip install -q ncps open-clip-torch tqdm torch torchvision

import sys
sys.path.insert(0, REPO_DIR)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# === Cellule 2 — Config chemins Drive ===
DRIVE_PATH = '/content/drive/MyDrive/sdnc/'
CHECKPOINT_DIR = DRIVE_PATH + 'checkpoints/'
LOG_DIR = DRIVE_PATH + 'logs/'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Logs: {LOG_DIR}')

In [ ]:
# === Cellule 3 — Reprise automatique ===
import glob
import json
from sdnc.config import SDNCConfig
from sdnc.model import SDNCModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = SDNCConfig()
model = SDNCModel(config).to(device)

# Find latest checkpoint
checkpoints = sorted(glob.glob(CHECKPOINT_DIR + 'step_*.pt'))
start_step = 0

if checkpoints:
    latest = checkpoints[-1]
    print(f'Reprise depuis: {latest}')
    ckpt = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    start_step = ckpt['step'] + 1
    # Restore global state and temporal stream if saved
    if 'global_state' in ckpt:
        model.global_state.state.copy_(ckpt['global_state'])
    if 'temporal_buffer' in ckpt:
        model.temporal_stream.buffer.copy_(ckpt['temporal_buffer'])
        model.temporal_stream.write_idx.fill_(ckpt.get('temporal_write_idx', 0))
        model.temporal_stream.filled.fill_(ckpt.get('temporal_filled', 0))
    print(f'Step de reprise: {start_step}')
    print(f'Prediction error au checkpoint: {ckpt.get("last_pred_error", "N/A")}')
else:
    print('Aucun checkpoint trouvé — démarrage from scratch')

print(f'\nDevice: {device}')
print(f'Paramètres trainables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# === Cellule 4 — Training avec sauvegarde auto ===
import time
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

# Dataset — CIFAR-10
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711)),
])
trainset = torchvision.datasets.CIFAR10(
    root='/content/data', train=True, download=True, transform=transform
)

# Episode sampling
def sample_episode(dataset, n_way=5, k_shot=1, query_per_class=15):
    """Sample a few-shot episode."""
    classes = torch.randperm(10)[:n_way]
    support_imgs, support_labels = [], []
    query_imgs, query_labels = [], []

    for label_idx, cls in enumerate(classes):
        cls_indices = [i for i, (_, l) in enumerate(dataset) if l == cls.item()]
        # Fast: use targets directly
        cls_indices = (torch.tensor(dataset.targets) == cls.item()).nonzero().squeeze(-1)
        perm = cls_indices[torch.randperm(len(cls_indices))[:k_shot + query_per_class]]

        for i in range(k_shot):
            img, _ = dataset[perm[i].item()]
            support_imgs.append(img)
            support_labels.append(label_idx)
        for i in range(k_shot, min(k_shot + query_per_class, len(perm))):
            img, _ = dataset[perm[i].item()]
            query_imgs.append(img)
            query_labels.append(label_idx)

    return (
        torch.stack(support_imgs),
        torch.tensor(support_labels),
        torch.stack(query_imgs),
        torch.tensor(query_labels),
    )

# Training config
TOTAL_EPISODES = 2000
SAVE_EVERY = 50
LOG_EVERY = 10
N_WAY = config.n_way
K_SHOT = config.k_shot
QUERY_PER_CLASS = config.query_per_class

# Logging
log_file = LOG_DIR + f'train_log_{int(time.time())}.jsonl'
print(f'Log: {log_file}')

model.train()
# IMPORTANT: soft_reset — keep global state & stream, reset circuits only
model.soft_reset()

running_pred_error = 0.0
running_accuracy = 0.0

pbar = tqdm(range(start_step, TOTAL_EPISODES), desc='Training')
for step in pbar:
    # Sample episode
    s_imgs, s_labels, q_imgs, q_labels = sample_episode(
        trainset, N_WAY, K_SHOT, QUERY_PER_CLASS
    )
    s_imgs = s_imgs.to(device)
    s_labels = s_labels.to(device)
    q_imgs = q_imgs.to(device)
    q_labels = q_labels.to(device)

    # NOTE: NO model.reset() here — stream is continuous!
    # Only reset circuit hidden states per episode (soft reset)
    model.circuit_bank.reset_states()

    # Learn from support
    learn_result = model.learn(images=s_imgs, labels=s_labels)
    pred_error = learn_result['prediction_error_norm']

    # Evaluate on query
    with torch.no_grad():
        scores = model.recognize(
            support_labels=s_labels,
            support_kwargs={'images': s_imgs},
            query_kwargs={'images': q_imgs},
        )
        accuracy = (scores.argmax(dim=-1) == q_labels).float().mean().item()

    # Running averages
    running_pred_error = 0.9 * running_pred_error + 0.1 * pred_error
    running_accuracy = 0.9 * running_accuracy + 0.1 * accuracy

    pbar.set_postfix({
        'acc': f'{running_accuracy:.3f}',
        'pred_err': f'{running_pred_error:.4f}',
        'active': model.circuit_bank.activation_history.nonzero().shape[0],
    })

    # Log
    if step % LOG_EVERY == 0:
        stats = model.get_predictive_stats()
        log_entry = {
            'step': step,
            'accuracy': accuracy,
            'running_accuracy': running_accuracy,
            'prediction_error': float(pred_error),
            'running_pred_error': float(running_pred_error),
            'n_active_circuits': stats['n_active_circuits'],
            'mean_pred_error': stats['mean_prediction_error'],
            'global_state_norm': stats['global_state']['state_norm'],
            'stream_filled': stats['temporal_stream']['filled'],
        }
        with open(log_file, 'a') as f:
            f.write(json.dumps(log_entry) + '\n')

    # Save checkpoint to Drive
    if step > 0 and step % SAVE_EVERY == 0:
        ckpt_path = CHECKPOINT_DIR + f'step_{step:05d}.pt'
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'global_state': model.global_state.state.clone(),
            'temporal_buffer': model.temporal_stream.buffer.clone(),
            'temporal_write_idx': model.temporal_stream.write_idx.item(),
            'temporal_filled': model.temporal_stream.filled.item(),
            'last_pred_error': float(running_pred_error),
            'last_accuracy': float(running_accuracy),
            'config': vars(config),
        }, ckpt_path)
        print(f'\n  Saved: {ckpt_path} (acc={running_accuracy:.3f}, pred_err={running_pred_error:.4f})')

print(f'\nTraining terminé — acc finale: {running_accuracy:.3f}')

In [ ]:
# === Cellule 5 — Évaluation : chat flou vs chat net ===
# Le TEST fondamental du predictive coding:
# - Chat net → faible erreur de prédiction ("je connais ça")
# - Chat flou → haute erreur de prédiction ("c'est bizarre")
# - MAIS les deux devraient quand même dire "chat" → incertitude gérée

import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

testset = torchvision.datasets.CIFAR10(
    root='/content/data', train=False, download=True, transform=transform
)

# Collect cat images (class 3 in CIFAR-10)
cat_indices = (torch.tensor(testset.targets) == 3).nonzero().squeeze(-1)
n_test = min(20, len(cat_indices))
test_indices = cat_indices[torch.randperm(len(cat_indices))[:n_test]]

model.eval()
model.soft_reset()

results_clean = []
results_blurry = []

for idx in test_indices:
    img_clean, label = testset[idx.item()]
    img_clean = img_clean.unsqueeze(0).to(device)

    # Create blurry version
    raw_img = testset.data[idx.item()]  # (32, 32, 3) numpy
    from PIL import Image, ImageFilter
    pil_img = Image.fromarray(raw_img)
    pil_blurry = pil_img.filter(ImageFilter.GaussianBlur(radius=3))
    img_blurry = transform(pil_blurry).unsqueeze(0).to(device)

    # Test prediction error
    with torch.no_grad():
        err_clean = model.get_prediction_error(images=img_clean)
        err_blurry = model.get_prediction_error(images=img_blurry)

    results_clean.append(err_clean['prediction_error_norm'])
    results_blurry.append(err_blurry['prediction_error_norm'])

mean_clean = sum(results_clean) / len(results_clean)
mean_blurry = sum(results_blurry) / len(results_blurry)

print(f'Chat net  — erreur de prédiction moyenne: {mean_clean:.4f}')
print(f'Chat flou — erreur de prédiction moyenne: {mean_blurry:.4f}')
print(f'Ratio flou/net: {mean_blurry / (mean_clean + 1e-8):.2f}x')
print()

if mean_blurry > mean_clean:
    print('SUCCÈS: Le modèle est plus surpris par les chats flous!')
    print('→ L\'erreur de prédiction code l\'incertitude, pas l\'ignorance.')
else:
    print('Le modèle ne distingue pas encore net/flou via prediction error.')
    print('→ Continuer le training ou ajuster les hyperparamètres.')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Chat net', 'Chat flou'], [mean_clean, mean_blurry],
            color=['#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Prediction Error')
axes[0].set_title('Prediction Error: Net vs Flou')

axes[1].hist(results_clean, alpha=0.7, label='Net', color='#2ecc71', bins=10)
axes[1].hist(results_blurry, alpha=0.7, label='Flou', color='#e74c3c', bins=10)
axes[1].legend()
axes[1].set_xlabel('Prediction Error')
axes[1].set_title('Distribution')

plt.tight_layout()
plt.savefig(LOG_DIR + 'eval_clean_vs_blurry.png', dpi=150)
plt.show()
print(f'Plot sauvegardé: {LOG_DIR}eval_clean_vs_blurry.png')

In [ ]:
# === Cellule 6 — Diagnostic complet ===
print('=== État du modèle ===')
stats = model.get_predictive_stats()
for k, v in stats.items():
    if isinstance(v, dict):
        print(f'\n{k}:')
        for kk, vv in v.items():
            print(f'  {kk}: {vv}')
    else:
        print(f'{k}: {v}')

print('\n=== Wiring inter-circuits ===')
wiring = model.get_circuit_wiring_stats()
for k, v in wiring.items():
    print(f'{k}: {v}')

print(f'\n=== Circuits les plus actifs ===')
top_active = model.circuit_bank.activation_history.topk(10)
for i, (count, idx) in enumerate(zip(top_active.values, top_active.indices)):
    pred_err = model.circuit_bank.prediction_error_history[idx].item()
    print(f'  Circuit {idx.item()}: {count.item():.0f} activations, pred_error={pred_err:.4f}')